## Load Packages

In [1]:
suppressPackageStartupMessages({
    library(data.table) 
    library(dplyr) 
    library(ggplot2)
    library(Seurat)
    library(SingleCellExperiment)
    library(dplyr)
    library(celldex)
    library(SingleR)
    library(RColorBrewer)
    library(scran)
    library(ComplexHeatmap)
    library(reshape2)
    library(viridis)
})

## Set Data Locations and Load in Data

In [2]:
# set paths to data locations
io = list()
io$main = "/rds/project/rds-SDzz0CATGms/users/ltgh2/"
io$embryo_sce = file.path(io$main, "projects/09_extended_atlas_revisions/ivan_data_2_6_2023/embryo_sce.rds") 
io$metadata_cells = file.path(io$main, "projects/09_extended_atlas_revisions/ivan_data_2_6_2023/metadata_cells.csv")
io$pca_batch_corrected = file.path(io$main, "projects/09_extended_atlas_revisions/ivan_data_2_6_2023/pca_batch_corrected.csv")
io$umap_layout = file.path(io$main, "projects/09_extended_atlas_revisions/ivan_data_2_6_2023/umap_layout.csv") 
io$cell_type_markers = file.path(io$main, "projects/09_extended_atlas_revisions/outputs/celltype_markers_seurat.rds") 

io$ivan_cell_type_markers = file.path(io$main, "projects/09_extended_atlas_revisions/ivan_marker_genes/Collated_Marker_Gene_List_20_7_2023.csv") 

io$haem_canonical_landscape = file.path(io$main, "projects/09_extended_atlas_revisions/landscapes/haem_canonical_landscape.rds") 
io$haem_full_landscape = file.path(io$main, "projects/09_extended_atlas_revisions/landscapes/haem_full_landscape.rds") 
io$haem_endothelium_landscape = file.path(io$main, "projects/09_extended_atlas_revisions/landscapes/haem_endothelium_landscape.rds") 


In [3]:
# set the working directory
setwd(io$main)

In [4]:
io

$main
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2/"

$embryo_sce
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/ivan_data_2_6_2023/embryo_sce.rds"

$metadata_cells
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/ivan_data_2_6_2023/metadata_cells.csv"

$pca_batch_corrected
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/ivan_data_2_6_2023/pca_batch_corrected.csv"

$umap_layout
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/ivan_data_2_6_2023/umap_layout.csv"

$cell_type_markers
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/outputs/celltype_markers_seurat.rds"

$ivan_cell_type_markers
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/ivan_marker_genes/Collated_Marker_Gene_List_20_7_2023.csv"

$haem_canonical_landscape
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/landscapes/haem_canonical_landscape.rds"

$haem_full_landscape
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/landscapes/haem_full_landscape.rds"

$haem_endothelium_landscape
[1] "/rds/project/rds-SDzz0CATGms/users/ltgh2//projects/09_extended_atlas_revisions/landscapes/haem_endothelium_landscape.rds"

In [5]:
# load in the data 
embryo_sce <- readRDS(io$embryo_sce)
metadata_cells <- read.csv(io$metadata_cells)
pca_batch_corrected <- read.csv(io$pca_batch_corrected)
umap_layout <- read.csv(io$umap_layout)

In [6]:
#haem_canonical_landscape <- readRDS(io$haem_canonical_landscape.rds)
#haem_full_landscape <- readRDS(io$haem_full_landscape.rds)
haem_endothelium_landscape <- readRDS(io$haem_endothelium_landscape) 

## Normalize Data with Ivan's precomputed Size Factors and Log Normalize

In [6]:
embryo_sce

class: SingleCellExperiment 
dim: 27669 430339 
metadata(0):
assays(1): counts
rownames(27669): ENSMUSG00000051951 ENSMUSG00000089699 ...
  ENSMUSG00000096730 ENSMUSG00000095742
rowData names(2): id mgi_symbol
colnames(430339): cell_1 cell_2 ... ext_cell_351871 ext_cell_351872
colData names(13): cell sample ... celltype_PijuanSala2019
  celltype_extended_atlas
reducedDimNames(2): UMAP PCA
mainExpName: NULL
altExpNames(0):

In [7]:
# Extract the size factors from int_colData
size_factors <- as.vector(embryo_sce@int_colData$size_factor)
size_factors <- as.numeric(size_factors)
str(size_factors)

 num [1:430339] 0.567 1.179 0.891 1.227 1.382 ...


In [8]:
# Compute log-normalized counts using size factors
log_norm_counts_precomputed <- logNormCounts(embryo_sce, size.factors = size_factors)
#log_norm_counts <- logNormCounts(embryo_sce)

In [9]:
log_norm_counts_precomputed

class: SingleCellExperiment 
dim: 27669 430339 
metadata(0):
assays(2): counts logcounts
rownames(27669): ENSMUSG00000051951 ENSMUSG00000089699 ...
  ENSMUSG00000096730 ENSMUSG00000095742
rowData names(2): id mgi_symbol
colnames(430339): cell_1 cell_2 ... ext_cell_351871 ext_cell_351872
colData names(14): cell sample ... celltype_extended_atlas sizeFactor
reducedDimNames(2): UMAP PCA
mainExpName: NULL
altExpNames(0):

## Adjust Row Names

In [10]:
# Assuming the rowData contains the MGI symbols under the column "mgi_symbol"
# Assign the MGI symbols as new rownames
rownames(log_norm_counts_precomputed) <- rowData(log_norm_counts_precomputed)$id

In [11]:
embryo_seurat_precomputed <- as.Seurat(log_norm_counts_precomputed, counts = "counts", data = "logcounts")

Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from UMAP to UMAP_”
Warning message:
“All keys should be one or more alphanumeric characters followed by an underscore '_', setting key to UMAP_”
Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from PC to PC_”
Warning message:
“All keys should be one or more alphanumeric characters followed by an underscore '_', setting key to PC_”


In [12]:
dim(embryo_seurat_precomputed)

[1]  27669 430339

In [13]:
# investigate duplicated gene names
check.duplicates <- duplicated(embryo_seurat_precomputed@assays$originalexp@meta.features$mgi_symbol)
table(check.duplicates)

# Sample character vector of gene names
gene_names <- embryo_seurat_precomputed@assays$originalexp@meta.features$mgi_symbol

# Find duplicated gene names
duplicated_genes <- gene_names[duplicated(gene_names)]

# Print duplicated gene names
print(duplicated_genes)

check.duplicates
FALSE  TRUE 
27638    31 

 [1] "Gm15853"       "Gm16701"       "Gm2464"        "Schip1"       
 [5] "Hist2h2bb"     "Smim20"        "Dancr"         "Gbp6"         
 [9] "D130017N08Rik" "Umad1"         "Ccdc142"       "Atn1"         
[13] "Apoc2"         "U2af1l4"       "Itgam"         "Map2k7"       
[17] "Fbxw14"        "3110039M20Rik" "4930556M19Rik" "Pcdha11"      
[21] "Pcdhga8"       "Fam205a2"      "Ccl21b"        "Il11ra2"      
[25] "Ccl27a"        "Ccl21c"        "Gm3286"        "Ccl27a"       
[29] "Il11ra2"       "Ccl19"         "Ccl21a"       


In [14]:
# Adjust names to remove duplicates
embryo_seurat_precomputed@assays$originalexp@meta.features$mgi_symbol <- make.unique(embryo_seurat_precomputed@assays$originalexp@meta.features$mgi_symbol)
check <- duplicated(embryo_seurat_precomputed@assays$originalexp@meta.features$mgi_symbol)
table(check)

rownames(embryo_seurat_precomputed@assays$originalexp@counts) <- embryo_seurat_precomputed@assays$originalexp@meta.features$mgi_symbol
rownames(embryo_seurat_precomputed@assays$originalexp@data) <- embryo_seurat_precomputed@assays$originalexp@meta.features$mgi_symbol

check
FALSE 
27669 

## Scale the RNA Expression

In [15]:
# Scale the gene expression data
#embryo_seurat_precomputed_scaled <- ScaleData(embryo_seurat_precomputed)

Centering and scaling data matrix



In [15]:
embryo_seurat_precomputed_scaled <- embryo_seurat_precomputed

## Marker Genes

In [16]:
# Create a list of marker genes for different cell types
marker_genes_list <- list(
  "Allantois" = c("Hoxa10", "Hoxa11", "Tbx2", "Tbx4"),
  "Allantois endothelium" = c("Cdh5", "Hoxa11", "Plac1", "Vcam1"),
  "Amniotic ectoderm" = c("Aqp2", "Hand1", "Lrrn4", "Msx1", "Msx2", "Tagln", "Tfap2a", "Vtcn1", "Wnt6", "Wt1"),
  "Anterior cardiopharyngeal progenitors" = c("Isl1", "Tcf21", "Tlx1"),
  "Anterior Primitive Streak" = c("Fgf8", "Foxa2", "Gsc", "Hhex", "Lhx1", "Mixl1", "Otx2", "Pax7"),
  "Anterior somitic tissues" = c("Arg1", "Id2", "Id3", "Irx3", "Marcks", "Meis2", "Meox1", "Mest", "Pax1", "Pax9", "Prrx2", "Sox9", "Tcf15"),
  "Blood progenitors" = c("Cited4", "Gata1", "Lmo2", "Runx1"),
  "Branchial arch neural crest" = c("Dlx1", "Dlx2", "Dlx3", "Dlx4", "Dlx5", "Dlx6", "Hoxa2", "Mef2c"),
  "Cardiomyocytes FHF 1" = c("Gata4", "Nkx2-5", "Nr2f2", "Osr1", "Tbx5", "Wnt2"),
  "Cardiomyocytes FHF 2" = c("Gata4", "Nkx2-5", "Nr2f2", "Osr1", "Tbx5", "Wnt2"),
  "Cardiomyocytes SHF 1" = c("Fgf10", "Fgf3", "Fgf8", "Isl1", "Itbp3", "Mef2c", "Tbx1", "Tlx1"),
  "Cardiomyocytes SHF 2" = c("Fgf10", "Fgf3", "Fgf8", "Isl1", "Itbp3", "Mef2c", "Nrp2", "Tbx1", "Tbx5", "Tlx1", "Wnt2"),
  "Cardiopharyngeal progenitors" = c("Mesp1", "Nr2f2", "Osr1", "Tbx5", "Wnt2", "Wt1"),
  "Cardiopharyngeal progenitors FHF" = c("Smarcd3"),
  "Cardiopharyngeal progenitors SHF" = c("Isl1", "Tcf21", "Tlx1"),
  "Caudal epiblast" = c("Nkx1-2"),
  "Caudal mesoderm" = c("Cdx1", "Cdx2", "Gbx2", "Hes7", "Hoxb1"),
  "Chorioallantoic-derived erythroid progenitors" = c("Abcb4", "Gata1", "Hba-x", "Hbb-bs", "Hbb-bt", "Hbb-y", "Kel", "Klf1", "Myb", "Slc4a1", "Sox6"),
  "Cranial mesoderm" = c("Apoe", "Ebf1", "Foxl2", "Id2", "Id3", "Irx3", "Myl9", "Ndufa4l2", "Pitx2", "Six1"),
  "Dermomyotome" = c("Aldh1a2", "Cadm1", "Dmrt2", "Meis2", "Meox1", "Meox2", "Myf5", "Pax3", "Snail2"),
  "Dorsal hindbrain progenitors" = c("Msx1", "Msx3", "Wn1", "Zic1", "Zic2", "Zic5"),
  "Dorsal midbrain neurons" = c("Ascl1", "Dll3", "Elavl3", "Elavl4", "Gadd45g", "Hes6", "Neurod4", "Neurog1"),
  "Dorsal spinal cord progenitors" = c("Hoxb2", "Hoxd4", "Msx1", "Pax3", "Pax6", "Zic1"),
  "Early dorsal forebrain progenitors" = c("Fgf17", "Fgf8", "Lhx2", "Mest", "Otx2", "Six3", "Six6"),
  "Ectoderm" = c("Otx2", "Pou3f1", "Pou5f1", "Sox2"),
  "Embryo proper endothelium" = c("Cdh5", "Crabp2", "Etv2", "Fev", "Hoxa2", "Chd5", "Pecam1", "Dlk1", "Meg3"),
  "Embryo proper mesothelium" = c("Acta2", "Aldh1a2", "Ctgf", "Eng", "Fn1", "Gata4", "Gpm6a", "Mef2c", "Pdgfra", "Pdgfrb", "Postn", "Tbx18", "Tbx5", "Tcf21", "Upk1b", "Upk3b", "Vim", "Wt1"),
  "EMP" = c("Csf1r", "Kit", "Myb", "Runx1"),
  "Endocardium" = c("Nkx2-5", "Pecam1", "Sox17", "Vwf", "Id1", "Id3", "Hey"),
  "Endotome" = c("Cxcl12", "Pax3", "Meox1", "Foxc2", "Pdgfra", "Alcam", "Hlf"),
  "Epiblast" = c("Dnmt3b", "Epcam", "Pou5f1", "Utf1"),
  "Epicardium" = c("Aldh1a2", "Bnc1", "Itga4", "Krt8", "Scx", "Sema3d", "Sfrp", "Sparc", "Tbx18", "Tcf21", "Upk3b", "Vcam1", "Wt1"),
  "Epidermis" = c("Hapln1", "Krt14", "Krt5", "Sfn", "Trp63", "Wnt4"),
  "Erythroid" = c("Abcb4", "Gata1", "Hba-x", "Hbb-bs", "Hbb-bt", "Hbb-y", "Kel", "Klf1", "Myb", "Slc4a1", "Sox6"),
  "ExE ectoderm" = c("Ascl2", "Elf5", "Tfap2c"),
  "ExE endoderm" = c("Apoa2", "Apoe1", "Cystm1", "Emb", "Ttr"),
  "Foregut" = c("Foxa1", "Foxa2", "Gata6", "Hhex", "Hnf4a", "Hoxa1", "Irx1", "Prox1", "Ripply3", "Sfrp5", "Tbx3", "Ttr"),
  "Forelimb" = NA,
  "Frontonasal mesenchyme" = c("Alx1", "Alx3", "Alx4", "Prrx1", "Twist1"),
  "Gut tube" = c("Apela", "Foxa2", "Sox17", "Spink1"),
  "Haematoendothelial progenitors" = c("Etv2", "Kdr"),
  "Hindbrain floor plate" = c("Foxa2", "Nkx2-9", "Nkx6-1", "Ntn1", "Shh"),
  "Hindbrain neural progenitors" = c("Hes5", "Hoxb4", "Hoxd4", "Olig2", "Pax6"),
  "Hindgut" = c("Cdx2", "Cdx4", "Foxa2", "Hoxa7", "Hoxb8", "Hoxc8", "Shh", "Sox17", "Sp5"),
  "Intermediate mesoderm" = c("Osr1"),
  "Kidney primordium" = c("Eya1", "Gata3", "Hoxd11", "Lhx1", "Nrp1", "Osr1", "Pax2", "Pax8", "Sall1", "Sox11"),
  "Late dorsal forebrain progenitors" = c("Id3", "Pax6", "Rspo2", "Rspo3", "Wnt8b"),
  "Lateral plate mesoderm" = c("Cdx2", "Cdx4", "Foxf1", "Fzd4", "Kdr"),
  "Limb ectoderm" = NA,
  "Limb mesoderm" = c("Hand1", "Hand2", "Lbx1", "Pax7", "Pitx1", "Prrx1", "Tbx4", "Tbx5"),
  "Megakaryocyte progenitors" = c("Itga2b", "Mpl", "Myl9", "Pf4", "Plek", "Ppbp", "Runx1", "Treml1", "Vwf"),
  "MEP" = c("Cd34", "Gata1", "Gata2", "Itga2b", "Klf1", "Mpl", "Plek", "Runx1", "Vwf"),
  "Mesenchyme" = c("Ahnak", "Bmp4", "Col3a1", "Dlk1", "Igf2", "Itga4", "Krt18", "Krt8", "Lgals1", "Pmp22"),
  "Midbrain progenitors" = c("Pax6", "Sp5"),
  "Midbrain/Hindbrain boundary" = c("En1", "En2", "Fgf8", "Gbx2", "Otx2", "Pax2", "Pax5", "Pax8", "Wnt1"),
  "Midgut" = c("Gata6", "Nepn"),
  "Migratory neural crest" = c("Foxd3", "Pax3", "Sox10", "Sox9", "Tfap2b"),
  "Nascent mesoderm" = c("Lefty2", "Mesp1", "Mesp2"),
  "Neural tube" = c("Cdx2", "Cdx4", "Hoxa7", "Hoxa9", "Hoxaas3", "Hoxb2", "Hoxb5os", "Hoxb8", "Hoxc8", "Hoxc9", "Nkx1-2"),
  "NMPs" = c("Cdx1", "Cdx2", "Cdx4", "Cyp26a1", "Epha5", "Fgf17", "Fgf8", "Hes3", "Hoxb9", "Nkx1-2", "Rspo3", "T"),
  "NMPs/Mesoderm-biased" = c("Cdx1", "Cdx2", "Cdx4", "Cyp26a1", "Fgf17", "Fgf8", "Nkx1-2", "Rspo3", "T"),
  "Node" = c("Chrd", "Dynlrb2", "Foxa2", "Foxj1", "Noto", "Pifo", "T"),
  "Non-neural ectoderm" = c("Bmp2", "Cxcl12", "Dkk1", "Dlk1", "Krt genes", "Krt18", "Krt19", "Krt7", "Krt8", "Msx genes", "Msx2", "Mt1", "Mt2", "Pdgfa", "Tfap2a", "Tfap2b", "Wnt4", "Wnt6"),
  "Notochord" = c("Bicc1", "Foxa2", "Nog", "Noto", "Shh", "T"),
  "Optic vesicle" = c("Foxd1", "Lhx2", "Otx2", "Pax6", "Rax", "Six6"),
  "Otic neural progenitors" = c("Eya2", "Isl1", "Neurod1", "Neurog1", "Six1"),
  "Otic placode" = c("Dlx5", "Lmx1a", "Pax8", "Six1", "Sox9", "Tbx2"),
  "Paraxial mesoderm" = c("Meox1", "Tbx1", "Tcf15"),
  "Parietal endoderm" = c("Lamb1", "Plat", "Sparc"),
  "PGC" = c("Dnd1", "Dppa3", "Ifitm3", "Nanos3", "Pou5f1", "Tfap2c"),
  "Pharyngeal endoderm" = c("Fgf8", "Isl1", "Meis2", "Nkx2-3", "Prrx2", "Six1"),
  "Pharyngeal mesoderm" = c("Isl1", "Tcf21"),
  "Placodal ectoderm" = c("Dlx5", "Dlx6", "Foxg1", "Otx2", "Pitx1", "Six1", "Six3", "Six6", "Sox2"),
  "Posterior somitic tissues" = c("Cadm1", "Dll3", "Hes7", "Laptm4b", "Lef1", "Pcdh19", "Slc9a3r1"),
  "Presomitic mesoderm" = c("Msgn1", "T", "Tbx6"),
  "Primitive Streak" = c("Eomes", "Nanog"),
  "Sclerotome" = c("Aldh1a2", "Meis2", "Meox1", "Myf5"),
  "Somitic mesoderm" = c("Aldh1a2", "Dll1", "Tbx6"),
  "Spinal cord progenitors" = c("Cdx1", "Cdx2", "Cdx4", "Hoxa10", "Hoxa3", "Hoxa9", "Hoxaas3", "Hoxb1", "Hoxb4", "Hoxb5", "Hoxb5os", "Hoxb8", "Hoxb9", "Hoxc10", "Hoxc6", "Hoxc8", "Hoxc9", "Hoxd4", "Ncam1", "Nkx1-2", "Nkx2-1", "Pax6", "Sox2", "T"),
  "Surface ectoderm" = c("Foxg1", "Grhl2", "Grhl3", "Trp63"),
  "Thyroid primordium" = c("Foxa2", "Foxe1", "Ltbp1", "Nkx2-1", "Nkx2-5", "Pax8", "Sox2"),
  "Venous endothelium" = c("Cdh5", "Ephb4", "Foxc1", "Foxc2", "Hoxa4", "Nr2f2", "Prox1", "Zic2", "Mef2c", "Clec1b", "Cldn5"),
  "Ventral forebrain progenitors" = c("Lhx2", "Lhx5", "Nkx2-1", "Nkx2-9", "Ptch1", "Shh"),
  "Ventral hindbrain progenitors" = c("Foxb1", "Hoxa2", "Hoxd4", "Nkx2-9", "Nkx6-1", "Nkx6-2", "Olig2", "Sfrp1", "Sfrp2"),
  "Visceral endoderm" = c("Amot", "Dkk1", "Emb", "Krt19", "Spink1", "Ttr"),
  "YS endothelium" = c("Anxa2", "Cdh5", "Itga4", "Lgals1", "Sparc", "Lyve1"),
  "YS mesothelium" = c("Hoxc8", "Lum"),
  "YS mesothelium-derived endothelial progenitors" = c("Anxa2", "Cdh5", "Itga4", "Lgals1", "Sparc")
)


In [17]:
str(marker_genes_list)

List of 88
 $ Allantois                                     : chr [1:4] "Hoxa10" "Hoxa11" "Tbx2" "Tbx4"
 $ Allantois endothelium                         : chr [1:4] "Cdh5" "Hoxa11" "Plac1" "Vcam1"
 $ Amniotic ectoderm                             : chr [1:10] "Aqp2" "Hand1" "Lrrn4" "Msx1" ...
 $ Anterior cardiopharyngeal progenitors         : chr [1:3] "Isl1" "Tcf21" "Tlx1"
 $ Anterior Primitive Streak                     : chr [1:8] "Fgf8" "Foxa2" "Gsc" "Hhex" ...
 $ Anterior somitic tissues                      : chr [1:13] "Arg1" "Id2" "Id3" "Irx3" ...
 $ Blood progenitors                             : chr [1:4] "Cited4" "Gata1" "Lmo2" "Runx1"
 $ Branchial arch neural crest                   : chr [1:8] "Dlx1" "Dlx2" "Dlx3" "Dlx4" ...
 $ Cardiomyocytes FHF 1                          : chr [1:6] "Gata4" "Nkx2-5" "Nr2f2" "Osr1" ...
 $ Cardiomyocytes FHF 2                          : chr [1:6] "Gata4" "Nkx2-5" "Nr2f2" "Osr1" ...
 $ Cardiomyocytes SHF 1                          : chr [1:8

## List of Cell Types

In [18]:
#############################
######### MESODERM ##########
#############################

## list mesoderm posterior --> anterior along the primitive streak

Early_gastrula <- c("ExE ectoderm",
                    "Epiblast",
                    "Caudal epiblast",
                    "Primitive Streak",
                    "Nascent mesoderm",
                    "PGC")

yolksac_blood <- c("Haematoendothelial progenitors", 
                   "EMP", 
                   "MEP", 
                   "Megakaryocyte progenitors", 
                   "Blood progenitors", 
                   "Erythroid", 
                   "Chorioallantoic-derived erythroid progenitors")

endo <- c("YS endothelium",
          "Venous endothelium",
          "Embryo proper endothelium",
          "Allantois endothelium")
    
yolksac_other <- c("YS mesothelium-derived endothelial progenitors",
                   "YS mesothelium")

mesenchymal <- c("Embryo proper mesothelium",
                 "Mesenchyme")                   
                
allantois <- c("Allantois endothelium", 
               "Allantois")


In [108]:
#############################
######### MESODERM ##########
#############################

## list mesoderm posterior --> anterior along the primitive streak

Early_gastrula <- c("ExE ectoderm",
                    "Epiblast",
                    "Caudal epiblast",
                    "Primitive Streak",
                    "Nascent mesoderm",
                    "PGC")

yolksac_blood <- c("Haematoendothelial progenitors", 
                   "EMP", 
                   "MEP", 
                   "Megakaryocyte progenitors", 
                   "Blood progenitors", 
                   "Erythroid", 
                   "Chorioallantoic-derived erythroid progenitors")

endo <- c("YS endothelium",
          "Venous endothelium",
          "Embryo proper endothelium",
          "Allantois endothelium")
    
yolksac_other <- c("YS mesothelium-derived endothelial progenitors",
                   "YS mesothelium")

yolksac_other_2 <- c("YS mesothelium-derived endothelial progenitors"
                   #"YS mesothelium"
                     )

mesenchymal <- c("Embryo proper mesothelium",
                 "Mesenchyme")                   
                
allantois <- c("Allantois" 
               #"Allantois endothelium"
              )

ExE_mesoderm <- c(yolksac_blood, endo, yolksac_other, mesenchymal, allantois) 

Cranial_mesoderm <- c("Cranial mesoderm")

Cardiac_mesoderm <- c("Pharyngeal mesoderm",
                  "Cardiopharyngeal progenitors",
                  "Cardiopharyngeal progenitors SHF",
                  "Anterior cardiopharyngeal progenitors",
                  "Cardiopharyngeal progenitors FHF",
                  "Cardiomyocytes FHF 1",
                  "Cardiomyocytes FHF 2",
                  "Cardiomyocytes SHF 1",
                  "Cardiomyocytes SHF 2",
                  "Epicardium",
                  "Endocardium")

Lateral_plate_mesoderm <- c("Lateral plate mesoderm",
                           "Limb mesoderm",
                           "Forelimb")

Intermediate_mesoderm <- c("Intermediate mesoderm", "Kidney primordium")

Paraxial_mesoderm <- c("Paraxial mesoderm", 
                  "Presomitic mesoderm", 
                  "Somitic mesoderm",
                  "Anterior somitic tissues",
                  "Posterior somitic tissues",
                  "Dermomyotome", 
                  "Endotome", 
                  "Sclerotome")

Axial_mesoderm <- c("Caudal mesoderm", 
                  "NMPs", 
                  "NMPs/Mesoderm-biased",
                  "Node",
                  "Notochord")
                       
Frontonasal_mesenchyme <- c("Frontonasal mesenchyme")

mesoderm_list <- c(Early_gastrula,
                   ExE_mesoderm, 
                   Cranial_mesoderm, 
                   Cardiac_mesoderm, 
                   Lateral_plate_mesoderm, 
                   Intermediate_mesoderm, 
                   Paraxial_mesoderm, 
                   Axial_mesoderm,
                   Frontonasal_mesenchyme)

non_ExE_mesoderm_list <- c(Early_gastrula,
                   Cranial_mesoderm, 
                   Cardiac_mesoderm, 
                   Lateral_plate_mesoderm, 
                   Intermediate_mesoderm, 
                   Paraxial_mesoderm, 
                   Axial_mesoderm,
                   Frontonasal_mesenchyme)

cardiac_somitic_mesoderm_list <- c(Cranial_mesoderm, 
                   Cardiac_mesoderm, 
                   Lateral_plate_mesoderm, 
                   Intermediate_mesoderm, 
                   Paraxial_mesoderm, 
                   Axial_mesoderm,
                   Frontonasal_mesenchyme)

hemato_endo <- c(yolksac_blood, 
                   endo, 
                   yolksac_other_2, 
                   allantois)

In [20]:
str(mesoderm_list)

 chr [1:53] "ExE ectoderm" "Epiblast" "Caudal epiblast" "Primitive Streak" ...


In [21]:
#############################
######### ECTODERM ##########
#############################

Surface_ectoderm <- c("Amniotic ectoderm",
                  "Surface ectoderm",
                  "Epidermis",
                  "Placodal ectoderm",
                  "Otic placode",
                  "Otic neural progenitors",
                  "Limb ectoderm")

# list A --> P + D --> V
Neural_tube <- c("Neural tube",
                "Optic vesicle",
                "Early dorsal forebrain progenitors",
                "Late dorsal forebrain progenitors",
                "Ventral forebrain progenitors",
                "Midbrain progenitors",
                "Dorsal midbrain neurons",
                "Midbrain/Hindbrain boundary",
                "Dorsal hindbrain progenitors", 
                "Hindbrain floor plate",
                "Hindbrain neural progenitors",
                "Ventral hindbrain progenitors",
                "Dorsal spinal cord progenitors",
                "Spinal cord progenitors")

Neural_crest <- c("Migratory neural crest",
                "Branchial arch neural crest")

extra_ectoderm <- c("Ectoderm", "Non-neural ectoderm")

ectoderm_list <- c(Surface_ectoderm, Neural_tube, Neural_crest, extra_ectoderm)



In [22]:
#############################
######### ENDODERM ##########
#############################
## list endoderm early --> late, anterior --> posterior 

Primitive_endoderm <- c("Parietal endoderm", 
                     "ExE endoderm",
                     "Visceral endoderm")

Definitive_endoderm <- c("Anterior Primitive Streak",
                 "Gut tube",
                 "Foregut",
                 "Midgut",
                 "Hindgut",
                 "Pharyngeal endoderm",
                 "Thyroid primordium")

endoderm_list <- c(Definitive_endoderm, Primitive_endoderm)

In [23]:
str(ectoderm_list)

 chr [1:25] "Amniotic ectoderm" "Surface ectoderm" "Epidermis" ...


In [24]:
all_germlayer_list <- c(ectoderm_list, mesoderm_list, endoderm_list)

str(all_germlayer_list)

 chr [1:88] "Amniotic ectoderm" "Surface ectoderm" "Epidermis" ...


In [25]:
all_cells <- c("Allantois", "Allantois endothelium", "Amniotic ectoderm", "Anterior cardiopharyngeal progenitors",
               "Anterior Primitive Streak", "Anterior somitic tissues", "Blood progenitors",
               "Branchial arch neural crest", "Cardiomyocytes FHF 1", "Cardiomyocytes FHF 2", "Cardiomyocytes SHF 1",
               "Cardiomyocytes SHF 2", "Cardiopharyngeal progenitors", "Cardiopharyngeal progenitors FHF",
               "Cardiopharyngeal progenitors SHF", "Caudal epiblast", "Caudal mesoderm",
               "Chorioallantoic-derived erythroid progenitors", "Cranial mesoderm", "Dermomyotome",
               "Dorsal hindbrain progenitors", "Dorsal midbrain neurons", "Dorsal spinal cord progenitors",
               "Early dorsal forebrain progenitors", "Ectoderm", "Embryo proper endothelium",
               "Embryo proper mesothelium", "EMP", "Endocardium", "Endotome", "Epiblast", "Epicardium", "Epidermis",
               "Erythroid", "ExE ectoderm", "ExE endoderm", "Foregut", "Forelimb", "Frontonasal mesenchyme",
               "Gut tube", "Haematoendothelial progenitors", "Hindbrain floor plate", "Hindbrain neural progenitors",
               "Hindgut", "Intermediate mesoderm", "Kidney primordium", "Late dorsal forebrain progenitors",
               "Lateral plate mesoderm", "Limb ectoderm", "Limb mesoderm", "Megakaryocyte progenitors", "MEP",
               "Mesenchyme", "Midbrain progenitors", "Midbrain/Hindbrain boundary", "Midgut", "Migratory neural crest",
               "Nascent mesoderm", "Neural tube", "NMPs", "NMPs/Mesoderm-biased", "Node", "Non-neural ectoderm",
               "Notochord", "Optic vesicle", "Otic neural progenitors", "Otic placode", "Paraxial mesoderm",
               "Parietal endoderm", "PGC", "Pharyngeal endoderm", "Pharyngeal mesoderm", "Placodal ectoderm",
               "Posterior somitic tissues", "Presomitic mesoderm", "Primitive Streak", "Sclerotome", "Somitic mesoderm",
               "Spinal cord progenitors", "Surface ectoderm", "Thyroid primordium", "Venous endothelium",
               "Ventral forebrain progenitors", "Ventral hindbrain progenitors", "Visceral endoderm",
               "YS endothelium", "YS mesothelium", "YS mesothelium-derived endothelial progenitors")

In [67]:
str(all_cells)

 chr [1:88] "Allantois" "Allantois endothelium" "Amniotic ectoderm" ...


In [69]:
# Split the character vector into four smaller vectors
split_size <- 22
char_vector_1 <- all_cells[1:split_size]
char_vector_2 <- all_cells[(split_size + 1):(2 * split_size)]
char_vector_3 <- all_cells[(2 * split_size + 1):(3 * split_size)]
char_vector_4 <- all_cells[(3 * split_size + 1):length(all_cells)]

# Print the smaller vectors
print(char_vector_1)
print(char_vector_2)
print(char_vector_3)
print(char_vector_4)

 [1] "Allantois"                                    
 [2] "Allantois endothelium"                        
 [3] "Amniotic ectoderm"                            
 [4] "Anterior cardiopharyngeal progenitors"        
 [5] "Anterior Primitive Streak"                    
 [6] "Anterior somitic tissues"                     
 [7] "Blood progenitors"                            
 [8] "Branchial arch neural crest"                  
 [9] "Cardiomyocytes FHF 1"                         
[10] "Cardiomyocytes FHF 2"                         
[11] "Cardiomyocytes SHF 1"                         
[12] "Cardiomyocytes SHF 2"                         
[13] "Cardiopharyngeal progenitors"                 
[14] "Cardiopharyngeal progenitors FHF"             
[15] "Cardiopharyngeal progenitors SHF"             
[16] "Caudal epiblast"                              
[17] "Caudal mesoderm"                              
[18] "Chorioallantoic-derived erythroid progenitors"
[19] "Cranial mesoderm"                       

In [70]:
not_in_both <- setdiff(unique(c(all_cells, all_germlayer_list)), intersect(all_cells, all_germlayer_list))

# Display the names that do not appear in both lists
print(not_in_both)

character(0)


## Marker Genes By Germ Layer

## Generate Heatmaps of Gene Expression

In [105]:
target_cell_type <- all_cells
target_markers <- unique(as.vector(unlist(marker_genes_list[target_cell_type])))
N <- 2
#plot_cells <- char_vector_1

# Variables to set to generate heatmaps
seurat <- embryo_seurat_precomputed_scaled
cells_to_plot <- target_cell_type
genes_to_plot <- target_markers

# Calculate average expression per gene and cell type
all_germ_layer_average_expression <- AverageExpression(seurat[,seurat@meta.data$celltype_extended_atlas %in% cells_to_plot], 
                                        features = genes_to_plot, 
                                        group.by = c("celltype_extended_atlas"),
                                        slot = "data")

all_germ_layer_expression_data <- melt(all_germ_layer_average_expression$originalexp, varnames = c("Gene", "Cell Type"))
range(all_germ_layer_expression_data$value)

# Variables to set to generate heatmaps
expression_data <- all_germ_layer_average_expression$originalexp
#expression_data <- ectoderm_expression_data
#expression_data <- endoderm_expression_data

cells_to_plot <- target_cell_type
#cells_to_plot <- ectoderm_list
#cells_to_plot <- endoderm_list

# Reorder the columns of the gene expression data based on the desired order
gene_expression_data_reordered <- expression_data[, cells_to_plot]

# Scale and center the gene expression data by row
scaled_centered_gene_expression_data <- t(apply(gene_expression_data_reordered, 1, scale))
colnames(scaled_centered_gene_expression_data) <- colnames(gene_expression_data_reordered)

# Function to get top N highly expressed genes for each cell type and remove duplicate gene names
get_top_genes_unique <- function(scaled_centered_gene_expression_data, n = N) {
  top_genes <- apply(scaled_centered_gene_expression_data, 2, function(x) {
    sorted_indices <- order(x, decreasing = TRUE)
    top_indices <- sorted_indices[1:n]
    top_genes <- rownames(scaled_centered_gene_expression_data)[top_indices]
    
    # Remove duplicate gene names
    unique_genes <- unique(top_genes)
    
    return(unique_genes)
  })
  return(top_genes)
}

# Get the top N most highly expressed unique genes for each cell type
top_genes_per_celltype <- get_top_genes_unique(scaled_centered_gene_expression_data, n = N)
top_genes_per_celltype <- unique(top_genes_per_celltype)

# Filter the original matrix to include only the top genes
filtered_gene_expression_data <- scaled_centered_gene_expression_data[top_genes_per_celltype, ]

# Transpose the filtered_gene_expression_data to switch rows and columns
transposed_data <- t(filtered_gene_expression_data)

# Remove columns with duplicate names
unique_matrix <- transposed_data[, !duplicated(colnames(transposed_data))]

# Set the desired scale limits
min_limit <- -1
max_limit <- 4

par(family = "Helvetica")  # Set the default font family to Helvetica

range(unique_matrix)

# Assuming unique_matrix is your data matrix
# Calculate the dimensions of the matrix
num_rows <- nrow(unique_matrix)
num_cols <- ncol(unique_matrix)

# Calculate appropriate width and height based on the dimensions
# You can adjust these factors as needed to fit your preferences
heatmap_width <- num_cols * 0.2
heatmap_height <- num_rows * 0.15

plot_size <- (22*N)
plot_genes_1 <- 1:(plot_size)
plot_genes_2 <- (plot_size+1):(2*plot_size)
plot_genes_3 <- (2*plot_size+1):(3*plot_size)
plot_genes_4 <- (3*plot_size+1):(4*plot_size)

# Create the heatmap using pheatmap with marker genes on the x-axis and cell types on the y-axis
heatmap_plot <- pheatmap(unique_matrix, 
         cluster_rows = FALSE,   # Invert the rows
         cluster_cols = FALSE,   # Invert the columns
         main = "Marker Gene Expression",
         color = viridis(100),
         fontsize_row = 5,
         fontsize_col = 5,
         border_color = FALSE,
         display_numbers = FALSE,
         breaks = seq(min_limit, max_limit, length.out = 101))

heatmap_plot


# Save the heatmap as a PDF file using base R graphics
output_filepath <- "projects/09_extended_atlas_revisions/code/final_plots/outputs/1_heatmap_all_cells_markers.pdf"
pdf(file = output_filepath, width = heatmap_width, height = heatmap_height)
print(heatmap_plot)
dev.off()  # Close the PDF graphics device

Warning message:
“The following 9 features were not found in the originalexp assay: Itbp3, Snail2, Wn1, Hey, Sfrp, Apoe1, NA, Krt genes, Msx genes”


[1]     0.00 68233.96

[1] -1.119303  9.274123

png 
  2

In [106]:
target_cell_type <- all_cells
target_markers <- unique(as.vector(unlist(marker_genes_list[target_cell_type])))
N <- 2
plot_cells <- char_vector_1

# Variables to set to generate heatmaps
seurat <- embryo_seurat_precomputed_scaled
cells_to_plot <- target_cell_type
genes_to_plot <- target_markers

# Calculate average expression per gene and cell type
all_germ_layer_average_expression <- AverageExpression(seurat[,seurat@meta.data$celltype_extended_atlas %in% cells_to_plot], 
                                        features = genes_to_plot, 
                                        group.by = c("celltype_extended_atlas"),
                                        slot = "data")

all_germ_layer_expression_data <- melt(all_germ_layer_average_expression$originalexp, varnames = c("Gene", "Cell Type"))
range(all_germ_layer_expression_data$value)

# Variables to set to generate heatmaps
expression_data <- all_germ_layer_average_expression$originalexp
#expression_data <- ectoderm_expression_data
#expression_data <- endoderm_expression_data

cells_to_plot <- target_cell_type
#cells_to_plot <- ectoderm_list
#cells_to_plot <- endoderm_list

# Reorder the columns of the gene expression data based on the desired order
gene_expression_data_reordered <- expression_data[, cells_to_plot]

# Scale and center the gene expression data by row
scaled_centered_gene_expression_data <- t(apply(gene_expression_data_reordered, 1, scale))
colnames(scaled_centered_gene_expression_data) <- colnames(gene_expression_data_reordered)

# Function to get top N highly expressed genes for each cell type and remove duplicate gene names
get_top_genes_unique <- function(scaled_centered_gene_expression_data, n = N) {
  top_genes <- apply(scaled_centered_gene_expression_data, 2, function(x) {
    sorted_indices <- order(x, decreasing = TRUE)
    top_indices <- sorted_indices[1:n]
    top_genes <- rownames(scaled_centered_gene_expression_data)[top_indices]
    
    # Remove duplicate gene names
    unique_genes <- unique(top_genes)
    
    return(unique_genes)
  })
  return(top_genes)
}

# Get the top N most highly expressed unique genes for each cell type
top_genes_per_celltype <- get_top_genes_unique(scaled_centered_gene_expression_data, n = N)
top_genes_per_celltype <- unique(top_genes_per_celltype)

# Filter the original matrix to include only the top genes
filtered_gene_expression_data <- scaled_centered_gene_expression_data[top_genes_per_celltype, ]

# Transpose the filtered_gene_expression_data to switch rows and columns
transposed_data <- t(filtered_gene_expression_data)

# Remove columns with duplicate names
unique_matrix <- transposed_data[, !duplicated(colnames(transposed_data))]

# Set the desired scale limits
min_limit <- -1
max_limit <- 4

par(family = "Helvetica")  # Set the default font family to Helvetica

range(unique_matrix)

# Assuming unique_matrix is your data matrix
# Calculate the dimensions of the matrix
num_rows <- nrow(unique_matrix)
num_cols <- ncol(unique_matrix)

# Calculate appropriate width and height based on the dimensions
# You can adjust these factors as needed to fit your preferences
heatmap_width <- num_cols * 0.2
heatmap_height <- num_rows * 0.15

plot_size <- (22*N)
plot_genes_1 <- 1:(plot_size)
plot_genes_2 <- (plot_size+1):(2*plot_size)
plot_genes_3 <- (2*plot_size+1):(3*plot_size)
plot_genes_4 <- (3*plot_size+1):(4*plot_size)

# Create the heatmap using pheatmap with marker genes on the x-axis and cell types on the y-axis
heatmap_plot <- pheatmap(unique_matrix[plot_cells,plot_genes_1], 
         cluster_rows = FALSE,   # Invert the rows
         cluster_cols = FALSE,   # Invert the columns
         main = "Marker Gene Expression",
         color = viridis(100),
         fontsize_row = 5,
         fontsize_col = 5,
         border_color = FALSE,
         display_numbers = FALSE,
         breaks = seq(min_limit, max_limit, length.out = 101))

heatmap_plot


# Save the heatmap as a PDF file using base R graphics
output_filepath <- "projects/09_extended_atlas_revisions/code/final_plots/outputs/1_heatmap_all_cells_1_markers.pdf"
pdf(file = output_filepath, width = heatmap_width, height = heatmap_height)
print(heatmap_plot)
dev.off()  # Close the PDF graphics device

Warning message:
“The following 9 features were not found in the originalexp assay: Itbp3, Snail2, Wn1, Hey, Sfrp, Apoe1, NA, Krt genes, Msx genes”


[1]     0.00 68233.96

[1] -1.119303  9.274123

png 
  2

In [102]:
target_cell_type <- all_cells
target_markers <- unique(as.vector(unlist(marker_genes_list[target_cell_type])))
N <- 2
plot_cells <- char_vector_2

# Variables to set to generate heatmaps
seurat <- embryo_seurat_precomputed_scaled
cells_to_plot <- target_cell_type
genes_to_plot <- target_markers

# Calculate average expression per gene and cell type
all_germ_layer_average_expression <- AverageExpression(seurat[,seurat@meta.data$celltype_extended_atlas %in% cells_to_plot], 
                                        features = genes_to_plot, 
                                        group.by = c("celltype_extended_atlas"),
                                        slot = "data")

all_germ_layer_expression_data <- melt(all_germ_layer_average_expression$originalexp, varnames = c("Gene", "Cell Type"))
range(all_germ_layer_expression_data$value)

# Variables to set to generate heatmaps
expression_data <- all_germ_layer_average_expression$originalexp
#expression_data <- ectoderm_expression_data
#expression_data <- endoderm_expression_data

cells_to_plot <- target_cell_type
#cells_to_plot <- ectoderm_list
#cells_to_plot <- endoderm_list

# Reorder the columns of the gene expression data based on the desired order
gene_expression_data_reordered <- expression_data[, cells_to_plot]

# Scale and center the gene expression data by row
scaled_centered_gene_expression_data <- t(apply(gene_expression_data_reordered, 1, scale))
colnames(scaled_centered_gene_expression_data) <- colnames(gene_expression_data_reordered)

# Function to get top N highly expressed genes for each cell type and remove duplicate gene names
get_top_genes_unique <- function(scaled_centered_gene_expression_data, n = N) {
  top_genes <- apply(scaled_centered_gene_expression_data, 2, function(x) {
    sorted_indices <- order(x, decreasing = TRUE)
    top_indices <- sorted_indices[1:n]
    top_genes <- rownames(scaled_centered_gene_expression_data)[top_indices]
    
    # Remove duplicate gene names
    unique_genes <- unique(top_genes)
    
    return(unique_genes)
  })
  return(top_genes)
}

# Get the top N most highly expressed unique genes for each cell type
top_genes_per_celltype <- get_top_genes_unique(scaled_centered_gene_expression_data, n = N)
top_genes_per_celltype <- unique(top_genes_per_celltype)

# Filter the original matrix to include only the top genes
filtered_gene_expression_data <- scaled_centered_gene_expression_data[top_genes_per_celltype, ]

# Transpose the filtered_gene_expression_data to switch rows and columns
transposed_data <- t(filtered_gene_expression_data)

# Remove columns with duplicate names
unique_matrix <- transposed_data[, !duplicated(colnames(transposed_data))]

# Set the desired scale limits
min_limit <- -1
max_limit <- 4

par(family = "Helvetica")  # Set the default font family to Helvetica

range(unique_matrix)

# Assuming unique_matrix is your data matrix
# Calculate the dimensions of the matrix
num_rows <- nrow(unique_matrix)
num_cols <- ncol(unique_matrix)

# Calculate appropriate width and height based on the dimensions
# You can adjust these factors as needed to fit your preferences
heatmap_width <- num_cols * 0.2
heatmap_height <- num_rows * 0.15

plot_size <- (22*N)
plot_genes_1 <- 1:(plot_size)
plot_genes_2 <- (plot_size+1):(2*plot_size-7)
plot_genes_3 <- (2*plot_size-7+1):(3*plot_size)
plot_genes_4 <- (3*plot_size+1):(4*plot_size)

# Create the heatmap using pheatmap with marker genes on the x-axis and cell types on the y-axis
heatmap_plot <- pheatmap(unique_matrix[plot_cells,plot_genes_2], 
         cluster_rows = FALSE,   # Invert the rows
         cluster_cols = FALSE,   # Invert the columns
         main = "Marker Gene Expression",
         color = viridis(100),
         fontsize_row = 5,
         fontsize_col = 5,
         border_color = FALSE,
         display_numbers = FALSE,
         breaks = seq(min_limit, max_limit, length.out = 101))

heatmap_plot


# Save the heatmap as a PDF file using base R graphics
output_filepath <- "projects/09_extended_atlas_revisions/code/final_plots/outputs/1_heatmap_all_cells_2_markers.pdf"
pdf(file = output_filepath, width = heatmap_width, height = heatmap_height)
print(heatmap_plot)
dev.off()  # Close the PDF graphics device

Warning message:
“The following 9 features were not found in the originalexp assay: Itbp3, Snail2, Wn1, Hey, Sfrp, Apoe1, NA, Krt genes, Msx genes”


[1]     0.00 68233.96

[1] -1.119303  9.274123

png 
  2

In [103]:
target_cell_type <- all_cells
target_markers <- unique(as.vector(unlist(marker_genes_list[target_cell_type])))
N <- 2
plot_cells <- char_vector_3

# Variables to set to generate heatmaps
seurat <- embryo_seurat_precomputed_scaled
cells_to_plot <- target_cell_type
genes_to_plot <- target_markers

# Calculate average expression per gene and cell type
all_germ_layer_average_expression <- AverageExpression(seurat[,seurat@meta.data$celltype_extended_atlas %in% cells_to_plot], 
                                        features = genes_to_plot, 
                                        group.by = c("celltype_extended_atlas"),
                                        slot = "data")

all_germ_layer_expression_data <- melt(all_germ_layer_average_expression$originalexp, varnames = c("Gene", "Cell Type"))
range(all_germ_layer_expression_data$value)

# Variables to set to generate heatmaps
expression_data <- all_germ_layer_average_expression$originalexp
#expression_data <- ectoderm_expression_data
#expression_data <- endoderm_expression_data

cells_to_plot <- target_cell_type
#cells_to_plot <- ectoderm_list
#cells_to_plot <- endoderm_list

# Reorder the columns of the gene expression data based on the desired order
gene_expression_data_reordered <- expression_data[, cells_to_plot]

# Scale and center the gene expression data by row
scaled_centered_gene_expression_data <- t(apply(gene_expression_data_reordered, 1, scale))
colnames(scaled_centered_gene_expression_data) <- colnames(gene_expression_data_reordered)

# Function to get top N highly expressed genes for each cell type and remove duplicate gene names
get_top_genes_unique <- function(scaled_centered_gene_expression_data, n = N) {
  top_genes <- apply(scaled_centered_gene_expression_data, 2, function(x) {
    sorted_indices <- order(x, decreasing = TRUE)
    top_indices <- sorted_indices[1:n]
    top_genes <- rownames(scaled_centered_gene_expression_data)[top_indices]
    
    # Remove duplicate gene names
    unique_genes <- unique(top_genes)
    
    return(unique_genes)
  })
  return(top_genes)
}

# Get the top N most highly expressed unique genes for each cell type
top_genes_per_celltype <- get_top_genes_unique(scaled_centered_gene_expression_data, n = N)
top_genes_per_celltype <- unique(top_genes_per_celltype)

# Filter the original matrix to include only the top genes
filtered_gene_expression_data <- scaled_centered_gene_expression_data[top_genes_per_celltype, ]

# Transpose the filtered_gene_expression_data to switch rows and columns
transposed_data <- t(filtered_gene_expression_data)

# Remove columns with duplicate names
unique_matrix <- transposed_data[, !duplicated(colnames(transposed_data))]

# Set the desired scale limits
min_limit <- -1
max_limit <- 4

par(family = "Helvetica")  # Set the default font family to Helvetica

range(unique_matrix)

# Assuming unique_matrix is your data matrix
# Calculate the dimensions of the matrix
num_rows <- nrow(unique_matrix)
num_cols <- ncol(unique_matrix)

# Calculate appropriate width and height based on the dimensions
# You can adjust these factors as needed to fit your preferences
heatmap_width <- num_cols * 0.2
heatmap_height <- num_rows * 0.15

plot_size <- (22*N)
plot_genes_1 <- 1:(plot_size)
plot_genes_2 <- (plot_size+1):((2*plot_size)-7)
plot_genes_3 <- (2*plot_size-7+1):(3*plot_size-14)
plot_genes_4 <- (3*plot_size-14+1):(4*plot_size)

# Create the heatmap using pheatmap with marker genes on the x-axis and cell types on the y-axis
heatmap_plot <- pheatmap(unique_matrix[plot_cells,plot_genes_3], 
         cluster_rows = FALSE,   # Invert the rows
         cluster_cols = FALSE,   # Invert the columns
         main = "Marker Gene Expression",
         color = viridis(100),
         fontsize_row = 5,
         fontsize_col = 5,
         border_color = FALSE,
         display_numbers = FALSE,
         breaks = seq(min_limit, max_limit, length.out = 101))

heatmap_plot


# Save the heatmap as a PDF file using base R graphics
output_filepath <- "projects/09_extended_atlas_revisions/code/final_plots/outputs/1_heatmap_all_cells_3_markers.pdf"
pdf(file = output_filepath, width = heatmap_width, height = heatmap_height)
print(heatmap_plot)
dev.off()  # Close the PDF graphics device

Warning message:
“The following 9 features were not found in the originalexp assay: Itbp3, Snail2, Wn1, Hey, Sfrp, Apoe1, NA, Krt genes, Msx genes”


[1]     0.00 68233.96

[1] -1.119303  9.274123

png 
  2

In [104]:
target_cell_type <- all_cells
target_markers <- unique(as.vector(unlist(marker_genes_list[target_cell_type])))
N <- 2
plot_cells <- char_vector_4

# Variables to set to generate heatmaps
seurat <- embryo_seurat_precomputed_scaled
cells_to_plot <- target_cell_type
genes_to_plot <- target_markers

# Calculate average expression per gene and cell type
all_germ_layer_average_expression <- AverageExpression(seurat[,seurat@meta.data$celltype_extended_atlas %in% cells_to_plot], 
                                        features = genes_to_plot, 
                                        group.by = c("celltype_extended_atlas"),
                                        slot = "data")

all_germ_layer_expression_data <- melt(all_germ_layer_average_expression$originalexp, varnames = c("Gene", "Cell Type"))
range(all_germ_layer_expression_data$value)

# Variables to set to generate heatmaps
expression_data <- all_germ_layer_average_expression$originalexp
#expression_data <- ectoderm_expression_data
#expression_data <- endoderm_expression_data

cells_to_plot <- target_cell_type
#cells_to_plot <- ectoderm_list
#cells_to_plot <- endoderm_list

# Reorder the columns of the gene expression data based on the desired order
gene_expression_data_reordered <- expression_data[, cells_to_plot]

# Scale and center the gene expression data by row
scaled_centered_gene_expression_data <- t(apply(gene_expression_data_reordered, 1, scale))
colnames(scaled_centered_gene_expression_data) <- colnames(gene_expression_data_reordered)

# Function to get top N highly expressed genes for each cell type and remove duplicate gene names
get_top_genes_unique <- function(scaled_centered_gene_expression_data, n = N) {
  top_genes <- apply(scaled_centered_gene_expression_data, 2, function(x) {
    sorted_indices <- order(x, decreasing = TRUE)
    top_indices <- sorted_indices[1:n]
    top_genes <- rownames(scaled_centered_gene_expression_data)[top_indices]
    
    # Remove duplicate gene names
    unique_genes <- unique(top_genes)
    
    return(unique_genes)
  })
  return(top_genes)
}

# Get the top N most highly expressed unique genes for each cell type
top_genes_per_celltype <- get_top_genes_unique(scaled_centered_gene_expression_data, n = N)
top_genes_per_celltype <- unique(top_genes_per_celltype)

# Filter the original matrix to include only the top genes
filtered_gene_expression_data <- scaled_centered_gene_expression_data[top_genes_per_celltype, ]

# Transpose the filtered_gene_expression_data to switch rows and columns
transposed_data <- t(filtered_gene_expression_data)

# Remove columns with duplicate names
unique_matrix <- transposed_data[, !duplicated(colnames(transposed_data))]

# Set the desired scale limits
min_limit <- -1
max_limit <- 4

par(family = "Helvetica")  # Set the default font family to Helvetica

range(unique_matrix)

# Assuming unique_matrix is your data matrix
# Calculate the dimensions of the matrix
num_rows <- nrow(unique_matrix)
num_cols <- ncol(unique_matrix)

# Calculate appropriate width and height based on the dimensions
# You can adjust these factors as needed to fit your preferences
heatmap_width <- num_cols * 0.2
heatmap_height <- num_rows * 0.15

plot_size <- (22*N)
plot_genes_1 <- 1:(plot_size)
plot_genes_2 <- (plot_size+1):((2*plot_size)-7)
plot_genes_3 <- (2*plot_size-7+1):(3*plot_size-14)
plot_genes_4 <- (3*plot_size-14+1):(ncol(unique_matrix))

# Create the heatmap using pheatmap with marker genes on the x-axis and cell types on the y-axis
heatmap_plot <- pheatmap(unique_matrix[plot_cells,plot_genes_4], 
         cluster_rows = FALSE,   # Invert the rows
         cluster_cols = FALSE,   # Invert the columns
         main = "Marker Gene Expression",
         color = viridis(100),
         fontsize_row = 5,
         fontsize_col = 5,
         border_color = FALSE,
         display_numbers = FALSE,
         breaks = seq(min_limit, max_limit, length.out = 101))

heatmap_plot


# Save the heatmap as a PDF file using base R graphics
output_filepath <- "projects/09_extended_atlas_revisions/code/final_plots/outputs/1_heatmap_all_cells_4_markers.pdf"
pdf(file = output_filepath, width = heatmap_width, height = heatmap_height)
print(heatmap_plot)
dev.off()  # Close the PDF graphics device

Warning message:
“The following 9 features were not found in the originalexp assay: Itbp3, Snail2, Wn1, Hey, Sfrp, Apoe1, NA, Krt genes, Msx genes”


[1]     0.00 68233.96

[1] -1.119303  9.274123

png 
  2

In [111]:
target_cell_type <- hemato_endo
target_markers <- unique(as.vector(unlist(marker_genes_list[target_cell_type])))
N <- 10
#plot_cells <- char_vector_4

# Variables to set to generate heatmaps
seurat <- embryo_seurat_precomputed_scaled
cells_to_plot <- target_cell_type
genes_to_plot <- target_markers

# Calculate average expression per gene and cell type
all_germ_layer_average_expression <- AverageExpression(seurat[,seurat@meta.data$celltype_extended_atlas %in% cells_to_plot], 
                                        features = genes_to_plot, 
                                        group.by = c("celltype_extended_atlas"),
                                        slot = "data")

all_germ_layer_expression_data <- melt(all_germ_layer_average_expression$originalexp, varnames = c("Gene", "Cell Type"))
range(all_germ_layer_expression_data$value)

# Variables to set to generate heatmaps
expression_data <- all_germ_layer_average_expression$originalexp
#expression_data <- ectoderm_expression_data
#expression_data <- endoderm_expression_data

cells_to_plot <- target_cell_type
#cells_to_plot <- ectoderm_list
#cells_to_plot <- endoderm_list

# Reorder the columns of the gene expression data based on the desired order
gene_expression_data_reordered <- expression_data[, cells_to_plot]

# Scale and center the gene expression data by row
scaled_centered_gene_expression_data <- t(apply(gene_expression_data_reordered, 1, scale))
colnames(scaled_centered_gene_expression_data) <- colnames(gene_expression_data_reordered)

# Function to get top N highly expressed genes for each cell type and remove duplicate gene names
get_top_genes_unique <- function(scaled_centered_gene_expression_data, n = N) {
  top_genes <- apply(scaled_centered_gene_expression_data, 2, function(x) {
    sorted_indices <- order(x, decreasing = TRUE)
    top_indices <- sorted_indices[1:n]
    top_genes <- rownames(scaled_centered_gene_expression_data)[top_indices]
    
    # Remove duplicate gene names
    unique_genes <- unique(top_genes)
    
    return(unique_genes)
  })
  return(top_genes)
}

# Get the top N most highly expressed unique genes for each cell type
top_genes_per_celltype <- get_top_genes_unique(scaled_centered_gene_expression_data, n = N)
top_genes_per_celltype <- unique(top_genes_per_celltype)

# Filter the original matrix to include only the top genes
filtered_gene_expression_data <- scaled_centered_gene_expression_data[top_genes_per_celltype, ]

# Transpose the filtered_gene_expression_data to switch rows and columns
transposed_data <- t(filtered_gene_expression_data)

# Remove columns with duplicate names
unique_matrix <- transposed_data[, !duplicated(colnames(transposed_data))]

# Set the desired scale limits
min_limit <- -1
max_limit <- 4

par(family = "Helvetica")  # Set the default font family to Helvetica

range(unique_matrix)

# Assuming unique_matrix is your data matrix
# Calculate the dimensions of the matrix
num_rows <- nrow(unique_matrix)
num_cols <- ncol(unique_matrix)

# Calculate appropriate width and height based on the dimensions
# You can adjust these factors as needed to fit your preferences
heatmap_width <- num_cols * 0.2
heatmap_height <- num_rows * 0.15

plot_size <- (22*N)
plot_genes_1 <- 1:(plot_size)
plot_genes_2 <- (plot_size+1):((2*plot_size)-7)
plot_genes_3 <- (2*plot_size-7+1):(3*plot_size-14)
plot_genes_4 <- (3*plot_size-14+1):(ncol(unique_matrix))

# Create the heatmap using pheatmap with marker genes on the x-axis and cell types on the y-axis
heatmap_plot <- pheatmap(unique_matrix, 
         cluster_rows = FALSE,   # Invert the rows
         cluster_cols = FALSE,   # Invert the columns
         main = "Marker Gene Expression",
         color = viridis(100),
         fontsize_row = 5,
         fontsize_col = 5,
         border_color = FALSE,
         display_numbers = FALSE,
         breaks = seq(min_limit, max_limit, length.out = 101))

heatmap_plot


# Save the heatmap as a PDF file using base R graphics
output_filepath <- "projects/09_extended_atlas_revisions/code/final_plots/outputs/1_hemato_endo_markers.pdf"
pdf(file = output_filepath, width = heatmap_width, height = heatmap_height)
print(heatmap_plot)
dev.off()  # Close the PDF graphics device

[1]     0.00 68233.96

[1] -1.203788  3.327005

png 
  2